## 03 — Node2Vec embeddings: setup + core utilities

This section does three things:
1) defines paths + reproducibility settings (load-first),
2) loads frozen co-occurrence edges and defines helpers (graph builder, embedding loader, train/load),
3) prepares the report runner (load + score saved iterations).


In [35]:
# 03 — Node2Vec Ingredient Embeddings (CLEAN / REPRODUCIBLE)

from pathlib import Path

# We keep all I/O under ../data/processed so the notebook stays portable.
# (Run this notebook from the notebooks/ folder for the relative paths to work.)
DATA_DIR = Path("../data/processed")
EMB_DIR  = DATA_DIR / "embeddings"

# Main frozen inputs:
# - COOC_EDGES_PATH is the co-occurrence graph edge list we train/evaluate against.
# - EVAL_SET_PATH is the fixed list of ingredients we score on (so runs are comparable).
COOC_EDGES_PATH = DATA_DIR / "ingredient_cooccurrence_edges.csv"
COOC_EDGES_ASSOC_PATH = DATA_DIR / "ingredient_cooccurrence_edges_assoc_strength.csv"  # only for assoc-weighted variants
BIPARTITE_EDGES_PATH = DATA_DIR / "ingredient_recipe_bipartite_edges.csv"              # only for bipartite experiments
EVAL_SET_PATH = DATA_DIR / "eval_ingredients_300_seed42.csv"                           # frozen eval set for the metric

# Where embedding .txt files live (load-first; training code should write here)
EMB_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Frozen evaluation protocol
# -----------------------------
# Keep these fixed across iterations so scores are apples-to-apples.
SEED = 42
TOPK = 10
RANK_SCHEME = "log"  # rank discounting scheme used by the metric

# -----------------------------
# Recompute policy
# -----------------------------
# The whole point of this notebook is reproducibility:
# if an output already exists, we reuse it instead of retraining.
SKIP_IF_EXISTS = True

# -----------------------------
# Sanity checks (minimum required files)
# -----------------------------
# Fail fast if the "frozen" inputs aren't there — otherwise you end up scoring nonsense.
required = [COOC_EDGES_PATH, EVAL_SET_PATH]
missing = [p for p in required if not p.exists()]

print("DATA_DIR:", DATA_DIR.resolve())
print("EMB_DIR :", EMB_DIR.resolve())

print("\nRequired inputs:")
for p in required:
    print("  ", "OK " if p.exists() else "MISS", p)

if missing:
    # Clear error message because this is the most common notebook setup issue.
    raise FileNotFoundError(
        "Missing required frozen inputs:\n" + "\n".join(f"- {p}" for p in missing) +
        "\n\nFix: ensure your ../data/processed folder contains these files."
    )

print("\nOptional inputs (used in some experiments):")
for p in [COOC_EDGES_ASSOC_PATH, BIPARTITE_EDGES_PATH]:
    print("  ", "OK " if p.exists() else "----", p)

print("\nConfig:")
print("  SEED =", SEED)
print("  TOPK =", TOPK, "| RANK_SCHEME =", RANK_SCHEME)
print("  SKIP_IF_EXISTS =", SKIP_IF_EXISTS)


DATA_DIR: /Users/adarpalaoglu/Documents/cs310-safeplate/data/processed
EMB_DIR : /Users/adarpalaoglu/Documents/cs310-safeplate/data/processed/embeddings

Required inputs:
   OK  ../data/processed/ingredient_cooccurrence_edges.csv
   OK  ../data/processed/eval_ingredients_300_seed42.csv

Optional inputs (used in some experiments):
   OK  ../data/processed/ingredient_cooccurrence_edges_assoc_strength.csv
   OK  ../data/processed/ingredient_recipe_bipartite_edges.csv

Config:
  SEED = 42
  TOPK = 10 | RANK_SCHEME = log
  SKIP_IF_EXISTS = True


### Core utilities (defined once, reused everywhere)

Shared helpers used by all experiments:
- load frozen co-occurrence edges
- build graphs only if training is needed
- load Word2Vec text embeddings safely
- single train-or-load entry point
- quick neighbour sanity checks


In [36]:
import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from gensim.models import KeyedVectors

# -----------------------------
# 1) Load frozen co-occurrence edges (read-only)
# -----------------------------

# Read the co-occurrence edge list into memory.
# This is treated as immutable for the rest of the notebook.
edges_df = pd.read_csv(COOC_EDGES_PATH)

# Defensive check: if this fails, we probably pointed to the wrong CSV.
assert {"ingredient_a", "ingredient_b", "weight"} <= set(edges_df.columns), \
    f"Unexpected schema in {COOC_EDGES_PATH}. Expected columns: ingredient_a, ingredient_b, weight"

# Force types explicitly to avoid pandas silently keeping object dtype.
# This matters later when building sparse matrices / graphs.
edges_df["ingredient_a"] = edges_df["ingredient_a"].astype(str)
edges_df["ingredient_b"] = edges_df["ingredient_b"].astype(str)
edges_df["weight"] = edges_df["weight"].astype(float)

# Quick sanity info so we know what graph we’re working with.
print("Loaded co-occurrence edges:", COOC_EDGES_PATH)
print(
    "Rows:", len(edges_df),
    "| unique ingredients:",
    pd.unique(
        pd.concat([edges_df["ingredient_a"], edges_df["ingredient_b"]])
    ).size
)

# -----------------------------
# 2) Graph builder (only call if training is required)
# -----------------------------

# This function converts the edge list into a NetworkX graph.
# Important: we only need this when *training* Node2Vec.
# If we are just loading saved embeddings, this never gets called.
def build_cooc_graph(edges_df: pd.DataFrame, weighted: bool = True) -> nx.Graph:
    """
    Build an undirected ingredient graph from the edge list.

    weighted=True  -> use edge weights to bias random walks
    weighted=False -> ignore weights (all edges treated equally)
    """
    # Undirected graph because co-occurrence is symmetric.
    G = nx.Graph()

    if weighted:
        # Add edges with an explicit weight attribute.
        # Node2Vec will use this to bias transition probabilities.
        G.add_weighted_edges_from(
            edges_df[["ingredient_a", "ingredient_b", "weight"]]
            .itertuples(index=False, name=None),
            weight="weight",
        )
    else:
        # Add edges without weights for unweighted random walks.
        G.add_edges_from(
            edges_df[["ingredient_a", "ingredient_b"]]
            .itertuples(index=False, name=None)
        )

    return G

# -----------------------------
# 3) Robust loader for Word2Vec text embeddings with tokens that may contain spaces
# -----------------------------

# Gensim’s default text loader can break when tokens contain spaces
# (e.g. "real butter"), so we parse the file manually.
def load_word2vec_text_allow_spaces(path: Path, encoding="utf-8") -> KeyedVectors:
    """
    Loads a Word2Vec *text* file where tokens may contain spaces.

    Assumed format:
      header: "<vocab_size> <dim>"
      line:   "<token possibly with spaces> <f1> ... <f_dim>"
    """
    # Ensure the file actually exists before doing anything else.
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Embeddings not found: {path}")

    # Open as plain text and parse line-by-line.
    with path.open("r", encoding=encoding, errors="replace") as f:
        # First line should be vocab size + vector dimension.
        header = f.readline().strip().split()
        if len(header) < 2:
            raise ValueError(f"Bad header in embeddings file: {path}")

        # Read dimension from header; vocab size is optional but informative.
        n_vocab = int(header[0])
        dim = int(header[1])

        # Initialise empty KeyedVectors container.
        kv = KeyedVectors(vector_size=dim)

        tokens = []
        vectors = []

        # Parse each embedding line manually.
        for line in f:
            line = line.rstrip("\n")

            # Skip empty lines defensively.
            if not line:
                continue

            parts = line.split(" ")

            # Need at least one token + dim numbers.
            if len(parts) < dim + 1:
                continue

            # Last `dim` values are the vector.
            vec_str = parts[-dim:]

            # Everything before that belongs to the token (may include spaces).
            tok_parts = parts[:-dim]
            token = " ".join(tok_parts).strip()

            # Convert vector to floats; skip malformed rows.
            try:
                vec = [float(x) for x in vec_str]
            except ValueError:
                continue

            # Only keep non-empty tokens.
            if token:
                tokens.append(token)
                vectors.append(vec)

        # Bulk add is much faster than adding one by one.
        kv.add_vectors(tokens, vectors)

    return kv

# -----------------------------
# 4) Train-or-load helper (single entry point for all iterations)
# -----------------------------

# This is the *only* function that should ever train Node2Vec.
# Everything else should go through this to avoid accidental retraining.
def get_embeddings_cooc_node2vec(
    *,
    iter_id: str,              # iteration ID used in filenames
    dims: int,                 # embedding dimension
    walk_length: int,          # length of each random walk
    num_walks: int,            # number of walks per node
    window: int,               # Word2Vec context window
    p: float,                  # Node2Vec return parameter
    q: float,                  # Node2Vec in-out parameter
    unweighted_walks: bool,    # whether to ignore edge weights
    seed: int,                 # RNG seed for reproducibility
    walk_workers: int,         # workers for random walk generation
    w2v_workers: int,          # workers for Word2Vec training
    epochs: int = 5,           # Word2Vec epochs
    batch_words: int = 128,    # Word2Vec batch size
    min_count: int = 1,        # keep all tokens
) -> KeyedVectors:
    """
    Returns KeyedVectors for a co-occurrence Node2Vec run.

    If SKIP_IF_EXISTS is True and the output file exists:
      -> load embeddings from disk
    Otherwise:
      -> train Node2Vec and save embeddings
    """

    # Encode whether walks were weighted directly into the filename.
    tag = "UNWEIGHTED" if unweighted_walks else "WEIGHTED"

    # Construct output filename deterministically from hyperparameters.
    out_path = EMB_DIR / (
        f"ingredient_node2vec_iter{iter_id}_{tag}_p{p}_q{q}_w{window}_"
        f"wl{walk_length}_r{num_walks}_d{dims}_e{epochs}.txt"
    )

    # Fast path: load existing embeddings if allowed.
    if SKIP_IF_EXISTS and out_path.exists():
        print(f"[LOAD] Iter {iter_id} from disk:", out_path)
        kv = load_word2vec_text_allow_spaces(out_path)
        print("Vectors:", len(kv), "| dim:", kv.vector_size)
        return kv

    # Slow path: actually train Node2Vec.
    print(f"[TRAIN] Iter {iter_id} -> {out_path.name}")

    # Build graph only when we really need it.
    G = build_cooc_graph(edges_df, weighted=(not unweighted_walks))

    print(
        "Graph:",
        "nodes =", G.number_of_nodes(),
        "| edges =", G.number_of_edges()
    )

    # Initialise Node2Vec with the chosen walk parameters.
    n2v = Node2Vec(
        G,
        dimensions=dims,
        walk_length=walk_length,
        num_walks=num_walks,
        p=p,
        q=q,
        weight_key=("weight" if not unweighted_walks else None),
        workers=walk_workers,
        seed=seed,
    )

    # Fit skip-gram model on the generated walks.
    model = n2v.fit(
        window=window,
        min_count=min_count,
        batch_words=batch_words,
        workers=w2v_workers,
        epochs=epochs,
    )

    # Save embeddings in text format so they’re easy to reload later.
    model.wv.save_word2vec_format(str(out_path))
    print("[SAVE]", out_path)
    print("Vectors:", len(model.wv), "| dim:", model.wv.vector_size)

    return model.wv

# -----------------------------
# 5) Fixed probe list (canonical tokens you used)
# -----------------------------

# Small, hand-picked ingredients used to quickly inspect neighbourhood quality.
# This is only for sanity checks, not for evaluation.
probe_ingredients = [
    "real butter",
    "pure olive oil",
    "milk",
    "flour",
    "chicken",
    "egg",
    "cream",
    "yogurt",
    "rice",
    "pasta",
]

# Utility to print nearest neighbours by cosine similarity.
# Helps catch obvious failures (missing tokens, junk neighbours).
def print_neighbours(kv: KeyedVectors, probes, topn=10, title=None):
    if title:
        print("\n" + "#" * 70)
        print(title)
        print("#" * 70)

    # Split probes into those present vs missing from the vocab.
    vocab = set(kv.key_to_index.keys())
    present = [p for p in probes if p in vocab]
    missing = [p for p in probes if p not in vocab]

    print("Present probes:", present)
    print("Missing probes:", missing)

    # Print cosine neighbours for each present probe.
    for p in present:
        print("\n" + "=" * 60)
        print("Probe:", p)
        for i, (nbr, sim) in enumerate(kv.most_similar(p, topn=topn), 1):
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")


Loaded co-occurrence edges: ../data/processed/ingredient_cooccurrence_edges.csv
Rows: 419913 | unique ingredients: 2698


### Report runner: load + score saved embeddings

Loads existing embedding files, builds the metric once, scores all iterations,
and outputs the final result tables (no training here).

In [37]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import coo_matrix

# 0) Expected globals from earlier cells
# Assumes DATA_DIR and load_word2vec_text_allow_spaces are already defined.

# Directory where all embedding files live
EMB_DIR = DATA_DIR / "embeddings"

# 1) Exact embedding filenames on disk
# Hard-coded on purpose to avoid guessing or retraining.
EMBEDDING_FILES = {
    "0":  "ingredient_node2vec_iter0.txt",
    "1":  "ingredient_node2vec_iter1_p1_q4.txt",
    "2":  "ingredient_node2vec_iter2_p1_q2_w5.txt",
    "3":  "ingredient_node2vec_iter3_UNWEIGHTED_p1_q2_w5.txt",
    "4":  "ingredient_node2vec_iter4_UNWEIGHTED_p1_q4_w5.txt",
    "5":  "ingredient_node2vec_iter5_ASSOC_WEIGHTED_p1_q4_w5.txt",
    "6":  "ingredient_node2vec_iter6_ASSOC_PRUNED_top50_LOGCOUNT_p1_q4_w5.txt",
    "7":  "ingredient_node2vec_iter7_UNWEIGHTED_p1_q4_w5_r40.txt",
    "8":  "ingredient_node2vec_iter8_UNWEIGHTED_p1_q4_w10_r40.txt",
    "9":  "ingredient_node2vec_iter9_UNWEIGHTED_p1_q4_w10_wl80_r40_e10.txt",
    "10": "ingredient_node2vec_iter10_UNWEIGHTED_p1_q4_w10_wl80_r80_e10.txt",
}

# Other embeddings exist but are not part of the main results.
# They can be added later if needed.
# "BIPPROJ_DEGNORM": ...
# "BIPARTITE_10K": ...

# 2) Load-only embedding access
# Strictly loads embeddings from disk; no training happens here.
def load_embeddings(iter_id: str):
    # Catch typos or unsupported iteration IDs early.
    if iter_id not in EMBEDDING_FILES:
        raise ValueError(f"iter_id={iter_id} not in EMBEDDING_FILES mapping.")

    path = EMB_DIR / EMBEDDING_FILES[iter_id]

    # Fail loudly if a claimed experiment file is missing.
    if not path.exists():
        raise FileNotFoundError(f"Missing embedding file for iter {iter_id}:\n  {path}")

    print(f"[LOAD] Iter {iter_id} -> {path.name}")

    # Custom loader handles ingredient tokens with spaces.
    kv = load_word2vec_text_allow_spaces(path)

    # Quick sanity info.
    print(f"  Vectors: {len(kv)} | dim: {kv.vector_size}")

    return kv, path


### Evaluation metric (built once, reused for all iterations)

Defines the rank-aware PPMI + co-use metric used to score substitutability.
This is expensive to build, so it is constructed once and cached in memory.


In [38]:
# 3) Metric definition (define once, only if missing)
# Avoid redefining the class if you re-run the cell.
if "CoocPPMIRankMetric" not in globals():
    class CoocPPMIRankMetric:
        """
        PairScore(i,j) = cosine(PPMI(i), PPMI(j)) * (1 - coUse(i,j))
        coUse(i,j) = max(P(j|i), P(i|j)) where P(j|i)=C(i,j)/freq(i)
        Rank-aware aggregation: log-discounted weighting by rank.
        """

        def __init__(self, edges_path: Path, eval_path: Path, seed=42, n_eval=300, keep_top_frac=0.70):
            self.edges_path = Path(edges_path)  # cooc edges used to build PPMI + co-use
            self.eval_path = Path(eval_path)    # where the frozen eval list lives
            self.seed = seed                    # eval sampling seed (if we have to create it)
            self.n_eval = n_eval                # expected size of eval list
            self.keep_top_frac = keep_top_frac  # only sample from frequent-ish ingredients

            # Filled in by build()
            self.edges_df = None
            self.vocab = None                   # index -> token
            self.idx = None                     # token -> index
            self.C = None                       # co-occurrence counts (sparse)
            self.marg = None                    # row sums of C
            self.P = None                       # PPMI matrix (sparse)
            self.norms = None                   # row norms for cosine
            self.eval_tokens = None             # list[str]

        def build(self, verbose=True):
            # Load the same cooc edges used for the graph (this metric is graph-derived).
            df = pd.read_csv(self.edges_path)
            df["ingredient_a"] = df["ingredient_a"].astype(str)
            df["ingredient_b"] = df["ingredient_b"].astype(str)
            df["weight"] = df["weight"].astype(float)
            self.edges_df = df

            # Build vocab from both endpoints.
            vocab = pd.Index(pd.unique(pd.concat([df["ingredient_a"], df["ingredient_b"]], ignore_index=True)))
            idx = {t: i for i, t in enumerate(vocab)}
            n = len(vocab)

            # Map tokens -> integer ids for sparse matrix building.
            ia = df["ingredient_a"].map(idx).to_numpy()
            ib = df["ingredient_b"].map(idx).to_numpy()
            w  = df["weight"].to_numpy()

            # Undirected: add both (a,b) and (b,a).
            rows = np.concatenate([ia, ib])
            cols = np.concatenate([ib, ia])
            data = np.concatenate([w,  w ])

            # C is the weighted co-occurrence matrix.
            C = coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()
            C.sum_duplicates()  # in case the edge list contains repeats

            # marg[i] = total co-occurrence weight for ingredient i.
            marg = np.array(C.sum(axis=1)).ravel()
            total = float(marg.sum())

            # Convert to COO to compute PMI only on non-zeros.
            Ccoo = C.tocoo()
            ii, jj, cij = Ccoo.row, Ccoo.col, Ccoo.data

            # denom = freq(i)*freq(j) in PMI formula.
            denom = marg[ii] * marg[jj]
            valid = (cij > 0) & (denom > 0)

            # Keep only valid entries (avoids /0 and junk).
            ii = ii[valid]; jj = jj[valid]; cij = cij[valid]; denom = denom[valid]

            # PMI(i,j) = log( (cij/total) / ((mi/total)*(mj/total)) ) -> simplifies to log(cij*total/(mi*mj))
            pmi = np.log((cij * total) / denom)

            # PPMI clips negatives to 0 (standard trick to keep only positive associations).
            ppmi = np.maximum(pmi, 0.0)

            # Sparse PPMI matrix (same shape as C).
            P = coo_matrix((ppmi, (ii, jj)), shape=(n, n)).tocsr()
            P.sum_duplicates()

            # Precompute row norms so cosine is cheap.
            norms = np.sqrt(P.multiply(P).sum(axis=1)).A1
            norms[norms == 0] = 1e-12  # avoid divide-by-zero

            # Store everything for later scoring.
            self.vocab = vocab
            self.idx = idx
            self.C = C
            self.marg = marg
            self.P = P
            self.norms = norms

            # Load eval list if it exists; otherwise generate and save it.
            self.eval_tokens = self._load_or_build_eval()

            if verbose:
                print("Metric built (PPMI + co-use penalty).")
                print("  Vocab size:", len(vocab))
                print("  PPMI nnz:", P.nnz)
                print("  Eval size:", len(self.eval_tokens))
            return self

        def _build_eval(self):
            # We sample eval ingredients from frequent-ish tokens to avoid super-rare noise.
            rng = np.random.default_rng(self.seed)

            # Sort by frequency proxy (marginal sums of C).
            order = np.argsort(-self.marg)
            tokens_sorted = self.vocab[order].to_numpy()

            # Keep only the top fraction by frequency.
            cutoff = int(len(tokens_sorted) * self.keep_top_frac)
            tokens = tokens_sorted[:cutoff]

            # Split into head/mid/tail bands for coverage.
            n = len(tokens)
            head_end = max(1, int(0.05 * n))
            mid_end  = max(head_end + 1, int(0.30 * n))

            head = tokens[:head_end]
            mid  = tokens[head_end:mid_end]
            tail = tokens[mid_end:]

            # Small helper to sample without replacement.
            def sample(arr, k):
                if len(arr) <= k:
                    return list(arr)
                return list(rng.choice(arr, size=k, replace=False))

            # 100 from each band (head/mid/tail).
            eval_tokens = sample(head, 100) + sample(mid, 100) + sample(tail, 100)

            # Remove duplicates while keeping order.
            eval_tokens = list(dict.fromkeys(eval_tokens))

            # If we ended up short (tiny dataset edge case), top up from the pool.
            if len(eval_tokens) < self.n_eval:
                pool = list(tokens)
                rng.shuffle(pool)
                have = set(eval_tokens)
                for t in pool:
                    if t not in have:
                        eval_tokens.append(t)
                        have.add(t)
                    if len(eval_tokens) >= self.n_eval:
                        break

            # Enforce exact length + shuffle to avoid any ordering bias.
            eval_tokens = eval_tokens[:self.n_eval]
            rng.shuffle(eval_tokens)
            return eval_tokens

        def _load_or_build_eval(self):
            p = self.eval_path

            # If a frozen eval file exists and matches the expected size, just use it.
            if p.exists():
                df = pd.read_csv(p)
                tokens = df["ingredient"].astype(str).tolist()
                if len(tokens) == self.n_eval:
                    return tokens

            # Otherwise, generate and write it so future runs are consistent.
            tokens = self._build_eval()
            pd.DataFrame({"ingredient": tokens}).to_csv(p, index=False)
            return tokens

        def _ppmi_cosine(self, a, b):
            # Return 0 if either token isn't in the metric vocab.
            if a not in self.idx or b not in self.idx:
                return 0.0

            i = self.idx[a]; j = self.idx[b]

            # Sparse dot product between PPMI rows.
            dot = self.P[i].multiply(self.P[j]).sum()

            # Cosine = dot / (||a|| ||b||)
            return float(dot / (self.norms[i] * self.norms[j]))

        def _co_use(self, a, b):
            # Co-use is based on raw co-occurrence counts, not PPMI.
            if a not in self.idx or b not in self.idx:
                return 0.0

            i = self.idx[a]; j = self.idx[b]

            # C[i,j] is the co-occurrence weight between the two.
            cij = float(self.C[i, j])
            if cij <= 0:
                return 0.0

            mi = float(self.marg[i]); mj = float(self.marg[j])
            if mi <= 0 or mj <= 0:
                return 0.0

            # max(P(j|i), P(i|j)) where P(j|i)=C(i,j)/freq(i)
            return float(max(cij / mi, cij / mj))

        def pair_score(self, a, b):
            # Similar context + not typically used together -> good substitution candidate.
            sim = self._ppmi_cosine(a, b)
            pen = 1.0 - self._co_use(a, b)   # high co-use => lower score
            pen = max(0.0, min(1.0, pen))    # clamp for safety
            return float(sim * pen)

        @staticmethod
        def _rank_weights(k, scheme="log"):
            # Rank weights: higher ranks count more (like DCG).
            r = np.arange(1, k + 1, dtype=float)

            if scheme == "log":
                w = 1.0 / np.log2(r + 1.0)
            elif scheme == "reciprocal":
                w = 1.0 / r
            else:
                raise ValueError("scheme must be 'log' or 'reciprocal'")

            # Normalise so weights sum to 1 (easy to interpret).
            return w / w.sum()

        def score(self, kv, topk=10, rank_scheme="log", exclude_prefixes=("recipe::",), min_candidate_marg=0.0):
            # Vocabulary available in the embedding model.
            model_vocab = set(kv.key_to_index.keys())

            # Optional lookup for candidate frequency filtering (not used if min_candidate_marg=0).
            marg_lookup = {t: float(self.marg[self.idx[t]]) for t in model_vocab if t in self.idx}

            scored = []   # per-query aggregated scores
            missing = 0   # eval queries missing from model or metric vocab
            empty = 0     # queries where we couldn't form a topk list

            for q in self.eval_tokens:
                # Skip if query isn't present in both spaces.
                if q not in model_vocab or q not in self.idx:
                    missing += 1
                    continue

                # Grab more than topk to survive filtering.
                raw = kv.most_similar(q, topn=topk * 5)

                cands = []
                for cand, _ in raw:
                    if cand == q:
                        continue
                    # Drop any special tokens (e.g. recipe nodes) if they exist.
                    if any(str(cand).startswith(p) for p in exclude_prefixes):
                        continue
                    # Candidate must exist in metric vocab to score it.
                    if cand not in self.idx:
                        continue
                    # Optional: ignore candidates below some frequency threshold.
                    if min_candidate_marg > 0 and marg_lookup.get(cand, 0.0) < min_candidate_marg:
                        continue

                    cands.append(cand)
                    if len(cands) >= topk:
                        break

                # If we couldn't get any valid candidates, skip this query.
                if not cands:
                    empty += 1
                    continue

                # PairScore for each candidate.
                scores = np.array([self.pair_score(q, c) for c in cands], dtype=float)

                # Rank weights depend on how many candidates we actually kept.
                w = self._rank_weights(len(scores), scheme=rank_scheme)

                # Weighted sum is the query’s final contribution.
                scored.append(float(np.sum(w * scores)))

            # Mean across queries is the single metric number for the model.
            overall = float(np.mean(scored)) if scored else 0.0

            # Handy metadata for logs/tables.
            meta = {
                "eval_set_size": len(self.eval_tokens),
                "scored_queries": len(scored),
                "missing_queries": missing,
                "empty_lists": empty,
                "topk": topk,
                "rank_scheme": rank_scheme,
            }
            return overall, scored, meta

# -----------------------------------------------------------
# 4) Build metric ONCE (expected to take time)
# -----------------------------------------------------------
# This is the expensive step (PPMI matrix build). Do it once, reuse for all iterations.
metric = CoocPPMIRankMetric(
    edges_path=COOC_EDGES_PATH,  # cooc edge list used for PPMI/co-use
    eval_path=EVAL_SET_PATH,     # frozen eval list path
    seed=SEED,                   # only used if eval list needs generating
    n_eval=300                   # expected eval size
).build(verbose=True)

# -----------------------------------------------------------
# 5) Report metadata (what you claim you ran)
# -----------------------------------------------------------
# Keep iterations grouped so tables are easy to interpret later.
COOC_FAMILY = [
    # early / baseline
    dict(iter_id="1",  family="cooc_tuning", notes="early baseline (p=1,q=4)"),
    dict(iter_id="2",  family="cooc_tuning", notes="tested less-DFS bias (q=2)"),
    dict(iter_id="3",  family="cooc_tuning", notes="unweighted walks w/ q=2"),
    dict(iter_id="4",  family="cooc_tuning", notes="baseline for main sweep (p=1,q=4,w=5)"),
    # main sweep
    dict(iter_id="7",  family="cooc_tuning", notes="more walks (r=40)"),
    dict(iter_id="8",  family="cooc_tuning", notes="larger window (w=10)"),
    dict(iter_id="9",  family="cooc_tuning", notes="longer walks + more epochs"),
    dict(iter_id="10", family="cooc_tuning", notes="more walks again (r=80)"),
]

GRAPH_VARIANTS = [
    dict(iter_id="5", family="graph_variant", notes="association-strength weighted graph"),
    dict(iter_id="6", family="graph_variant", notes="assoc-pruned (top50 logcount) graph"),
]

# Single list to iterate over in the scoring loop.
ALL_TO_SCORE = COOC_FAMILY + GRAPH_VARIANTS

# -----------------------------------------------------------
# 6) Score loop: LOAD -> score -> log -> neighbours
# -----------------------------------------------------------
results = []  # one row per iteration

def percentile_block(per_q):
    # Per-query score distribution is useful for sanity (not just the mean).
    if not per_q:
        return {}
    arr = np.array(per_q, dtype=float)
    return {f"p{p}": float(np.percentile(arr, p)) for p in [0,10,25,50,75,90,100]}

for cfg in ALL_TO_SCORE:
    iter_id = cfg["iter_id"]  # e.g. "8"

    # Load embeddings from disk (never train here).
    kv, emb_path = load_embeddings(iter_id)

    # Score this model on the frozen eval set.
    score, per_q, meta = metric.score(
        kv,
        topk=TOPK,                   # fixed across all runs
        rank_scheme=RANK_SCHEME,     # fixed across all runs
        exclude_prefixes=("recipe::",),
        min_candidate_marg=0.0,      # no extra filtering by default
    )

    # Log (kept consistent so you can skim outputs quickly).
    print("\n" + "=" * 88)
    print(f"ITERATION {iter_id} — SCORE LOG")
    print(f"- Family: {cfg['family']} | Notes: {cfg['notes']}")
    print(
        f"- Metric: ModelScore={score:.6f} | scored={meta['scored_queries']}/{meta['eval_set_size']} "
        f"| missing={meta['missing_queries']} | empty={meta['empty_lists']}"
    )
    if per_q:
        pct = percentile_block(per_q)
        print("- Per-query percentiles:", ", ".join([f"{k}={v:.6f}" for k, v in pct.items()]))
    print("- Embeddings file:", emb_path.name)
    print("=" * 88)

    # Quick “does this look sane?” neighbour printout.
    print_neighbours(kv, probe_ingredients, topn=10, title=f"Iter {iter_id} — cosine neighbours (sanity)")

    # Row for the final results table.
    row = {
        "iter": iter_id,
        "family": cfg["family"],
        "notes": cfg["notes"],
        "score": float(score),
        "scored_queries": meta["scored_queries"],
        "missing_queries": meta["missing_queries"],
        "empty_lists": meta["empty_lists"],
        "emb_file": emb_path.name,
        "emb_path": str(emb_path),
        "rank_scheme": RANK_SCHEME,
        "topk": TOPK,
        "seed": SEED,
    }

    # Add percentiles as extra columns (optional but handy).
    row.update(percentile_block(per_q))

    results.append(row)

# -----------------------------------------------------------
# 7) Report tables (two tables, cleaner story)
# -----------------------------------------------------------
results_df = pd.DataFrame(results)  # one row per iteration

# Split by experiment family for readability.
cooc_df = (
    results_df[results_df["family"] == "cooc_tuning"]
    .copy()
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)
var_df  = (
    results_df[results_df["family"] == "graph_variant"]
    .copy()
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

# Print + display as two separate tables.
print("\n" + "#" * 88)
print("TABLE A — Co-occurrence Node2Vec tuning (Iter 1,2,3,4,7,8,9,10)")
print("#" * 88)
display(cooc_df)

print("\n" + "#" * 88)
print("TABLE B — Graph variants (Iter 5,6)")
print("#" * 88)
display(var_df)

# Pick the best cooc config for the “main” model.
best = cooc_df.iloc[0]

print("\nBEST (cooc_tuning) BY METRIC:")
print(best[["iter", "score", "notes", "emb_file"]])

# Save tables so the report can pull them directly.
OUT_A = DATA_DIR / "node2vec_results_cooc_tuning_seed42.csv"
OUT_B = DATA_DIR / "node2vec_results_graph_variants_seed42.csv"

cooc_df.to_csv(OUT_A, index=False)
var_df.to_csv(OUT_B, index=False)

print("\nSaved report tables:")
print(" -", OUT_A)
print(" -", OUT_B)


Metric built.
  Vocab size: 2698
  PPMI nnz: 839826
  Eval size: 300
[LOAD] Iter 1 -> ingredient_node2vec_iter1_p1_q4.txt
  Vectors: 2698 | dim: 128

ITERATION 1 — SCORE LOG
- Family: cooc_tuning | Notes: early baseline (p=1,q=4)
- Metric: ModelScore=0.261620 | scored=300/300 | missing=0 | empty=0
- Per-query percentiles: p0=0.066512, p10=0.139054, p25=0.190757, p50=0.266790, p75=0.324755, p90=0.378303, p100=0.523947
- Embeddings file: ingredient_node2vec_iter1_p1_q4.txt

######################################################################
Iter 1 — cosine neighbours (sanity)
######################################################################
Present probes: ['real butter', 'pure olive oil', 'milk', 'flour', 'chicken', 'egg', 'cream', 'yogurt', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. jumbo egg                      cosine=0.639
02. vanilla caramel                cosine=0.626
03. farmer cheese                  cosine=0.614
04. cinnamon-swirl bread           cosine

,iter,family,notes,score,scored_queries,missing_queries,empty_lists,emb_file,emb_path,rank_scheme,topk,seed,p0,p10,p25,p50,p75,p90,p100
0,10,cooc_tuning,more walks again (r=80),0.388802,300,0,0,ingredient_node2vec_iter10_UNWEIGHTED_p1_q4_w1...,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.153721,0.221324,0.307211,0.384975,0.470201,0.528093,0.675979
1,9,cooc_tuning,longer walks + more epochs,0.383931,300,0,0,ingredient_node2vec_iter9_UNWEIGHTED_p1_q4_w10...,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.136867,0.215395,0.297684,0.384234,0.471259,0.530369,0.684479
2,8,cooc_tuning,larger window (w=10),0.340060,300,0,0,ingredient_node2vec_iter8_UNWEIGHTED_p1_q4_w10...,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.109120,0.184872,0.261385,0.346187,0.420593,0.480811,0.617457
3,7,cooc_tuning,more walks (r=40),0.295424,300,0,0,ingredient_node2vec_iter7_UNWEIGHTED_p1_q4_w5_...,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.109447,0.177580,0.234492,0.295980,0.354377,0.407364,0.510183
4,1,cooc_tuning,"early baseline (p=1,q=4)",0.261620,300,0,0,ingredient_node2vec_iter1_p1_q4.txt,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.066512,0.139054,0.190757,0.266790,0.324755,0.378303,0.523947
5,3,cooc_tuning,unweighted walks w/ q=2,0.247998,300,0,0,ingredient_node2vec_iter3_UNWEIGHTED_p1_q2_w5.txt,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.079434,0.142065,0.188090,0.242360,0.303760,0.355355,0.496233
6,2,cooc_tuning,tested less-DFS bias (q=2),0.247050,300,0,0,ingredient_node2vec_iter2_p1_q2_w5.txt,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.082851,0.144202,0.182866,0.238013,0.304086,0.352949,0.494055
7,4,cooc_tuning,"baseline for main sweep (p=1,q=4,w=5)",0.245978,300,0,0,ingredient_node2vec_iter4_UNWEIGHTED_p1_q4_w5.txt,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.085167,0.149580,0.183105,0.239617,0.298885,0.349402,0.503307



########################################################################################
TABLE B — Graph variants (Iter 5,6)
########################################################################################


,iter,family,notes,score,scored_queries,missing_queries,empty_lists,emb_file,emb_path,rank_scheme,topk,seed,p0,p10,p25,p50,p75,p90,p100
0,5,graph_variant,association-strength weighted graph,0.388052,300,0,0,ingredient_node2vec_iter5_ASSOC_WEIGHTED_p1_q4...,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.099490,0.199304,0.286379,0.397648,0.493846,0.550153,0.696744
1,6,graph_variant,assoc-pruned (top50 logcount) graph,0.342494,300,0,0,ingredient_node2vec_iter6_ASSOC_PRUNED_top50_L...,../data/processed/embeddings/ingredient_node2v...,log,10,42,0.107608,0.200432,0.256542,0.342141,0.421054,0.488745,0.674031



BEST (cooc_tuning) BY METRIC:
iter                                                       10
score                                                0.388802
notes                                 more walks again (r=80)
emb_file    ingredient_node2vec_iter10_UNWEIGHTED_p1_q4_w1...
Name: 0, dtype: object

Saved report tables:
 - ../data/processed/node2vec_results_cooc_tuning_seed42.csv
 - ../data/processed/node2vec_results_graph_variants_seed42.csv


### Stopping & selection rationale

- All iterations (1–10) are reported for transparency, but grouped by experiment family.
- **Table A (cooc_tuning)** is the main hyperparameter sweep on the same co-occurrence graph.
- **Table B (graph variants)** is reported separately to avoid mixing graph-construction effects.
- Iter10 is selected as the best co-occurrence configuration under the frozen, rank-aware metric
  (PPMI context similarity × co-use penalty with log-discounted ranks).
- Further tuning was stopped due to diminishing metric gains relative to compute cost.
- Final validation will be performed later in the **Synthetic Test Lab** with user-facing ranking.
